### Optimization

In [ ]:
# Optimization horizons
OPT_TRAIN_EPISODES = 52           # default matches Learning_DQN ponovitev
OPT_HORIZON = KorakovNaDan * 7   # matches Learning_DQN steps_per_episode


def objective(trial, train_episodes=None):
    if train_episodes is None:
        train_episodes = OPT_TRAIN_EPISODES

    # --- Hyperparameters (unchanged) ---
    gamma_t = trial.suggest_float("gamma", 0.8, 0.99999, log=True)
    lr_t = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    epsilon_start_t = trial.suggest_float("epsilon_start", 0.8, 1.0)
    epsilon_end_t = trial.suggest_float("epsilon_end", 0.01, 0.15)
    total_steps = train_episodes * OPT_HORIZON
    exploration_fraction = trial.suggest_float("exploration_fraction", 0.3, 0.8)
    steps_to_decay = total_steps * exploration_fraction
    epsilon_decay_t = (epsilon_start_t - epsilon_end_t) / steps_to_decay
    batch_size_t = trial.suggest_categorical("batch_size", [32, 64, 96, 128])
    fc1_t = 512
    fc2_t = 256
    fc3_t = 128
    replace_target_t = trial.suggest_int("replace_target", KorakovNaDan * 3, KorakovNaDan * 30)
    weight_decay_t = trial.suggest_float("weight_decay", 0.0, 1e-3)

    # --- Split train_data into opt_train (first half) and opt_val (second half) ---
    n = len(train_data)
    split_idx = n // 2
    opt_train_data = train_data.iloc[:split_idx]
    opt_train_data_norm = train_data_norm.iloc[:split_idx]
    opt_val_data = train_data.iloc[split_idx:]
    opt_val_data_norm = train_data_norm.iloc[split_idx:]

    # --- Phase 1: Train on opt_train with random reset ---
    train_env = build_dqn_env(
        dataset=opt_train_data,
        dataset_norm=opt_train_data_norm,
        episode_length=OPT_HORIZON,
        reset_mode="random",
        observation_mode="sliding_window",
    )

    state_shape = [int(train_env.observation_space.shape[0])]
    network_dims = [state_shape[0], fc1_t, fc2_t, fc3_t, len(Action)]

    agent = AgentDQN(
        gamma=gamma_t,
        epsilon=epsilon_start_t,
        lr=lr_t,
        state_shape=state_shape,
        network_dims=network_dims,
        batch_size=batch_size_t,
        eps_end=epsilon_end_t,
        eps_dec=epsilon_decay_t,
        weight_decay=weight_decay_t,
        replace_target=replace_target_t,
    )

    episode_rewards = []
    for episode in range(train_episodes):
        obs, _ = train_env.reset(options={"reset_mode": "random"})
        episode_reward = 0.0

        for _ in range(OPT_HORIZON):
            a = agent.choose_action(obs)
            next_obs, r, terminated, truncated, _ = train_env.step(a)
            done = bool(terminated or truncated)

            agent.store_transition(obs, a, r, next_obs, done)
            agent.learn()

            obs = next_obs
            episode_reward += r

            if done:
                break

        episode_rewards.append(episode_reward)

        # Report rolling average for pruning
        rolling_score = sum(episode_rewards) / len(episode_rewards)
        trial.report(rolling_score, episode)

        if trial.should_prune():
            raise optuna.TrialPruned()

    # --- Phase 2: Validate on opt_val with sequential reset (greedy) ---
    val_env = build_dqn_env(
        dataset=opt_val_data,
        dataset_norm=opt_val_data_norm,
        episode_length=len(opt_val_data) - 1,
        reset_mode="sequential",
        observation_mode="sliding_window",
    )

    old_epsilon = agent.epsilon
    agent.epsilon = 0.0

    obs, _ = val_env.reset(options={"reset_mode": "sequential"})
    val_reward = 0.0

    for _ in range(len(opt_val_data) - 1):
        a = agent.choose_action(obs)
        obs, r, terminated, truncated, _ = val_env.step(a)
        val_reward += r
        if terminated or truncated:
            break

    agent.epsilon = old_epsilon

    return val_reward


study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=20, n_warmup_steps=40),
)

print("Starting hyperparameter optimization...")
study.optimize(objective, n_trials=100, timeout=3600)

print("Optimization completed.")
print(f"Finished trials: {len(study.trials)}")
print(f"Best trial: {study.best_trial.number}")
print(f"Best value: {study.best_trial.value:.4f}")

best_params = study.best_trial.params

plot_optimization_history(study).show()
plot_param_importances(study).show()

In [ ]:
# apply best Optuna parameters
gamma = best_params['gamma']
lr = best_params['lr']
epsilon_start = best_params['epsilon_start']
epsilon_end = best_params['epsilon_end']
#epsilon_decay = best_params['epsilon_decay']
exploration_fraction = best_params['exploration_fraction']
total_steps = OPT_TRAIN_EPISODES * OPT_HORIZON
steps_to_decay = total_steps * exploration_fraction
epsilon_decay = (epsilon_start - epsilon_end) / steps_to_decay
batch_size = best_params['batch_size']
# fc1_dims = best_params['fc1_dims']
# fc2_dims = best_params['fc2_dims']
# fc3_dims = best_params['fc3_dims']
replace_target = best_params['replace_target']
weight_decay = best_params['weight_decay']